# Evaluating Cortex Agents: Hands-On Lab

**Duration:** 30-45 minutes  
**Scenario:** You're building a CMO-facing assistant that answers questions about campaign performance, budget allocation, and marketing strategy. The agent uses three tools: Cortex Analyst (structured data), Cortex Search (strategy documents), and an Agent Skill (executive summaries).

**What you'll do:**
1. Create the agent with all three tools
2. Build an evaluation dataset (12 questions across intent types)
3. Define metrics and run your first evaluation
4. Inspect results, identify weaknesses, and iterate
5. Version the improved agent and promote to production

**Prerequisites:** Run `setup.sql` before starting this notebook. It creates the database, tables, semantic view, Cortex Search service, and evaluation stage.

In [ ]:
# Connection setup — works in both Snowsight notebooks and local Jupyter
import os

try:
    # Snowsight notebook: session already exists
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local Jupyter: create session from environment or connection config
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
        "warehouse": "CMO_EVAL_WH",
        "database": "CMO_EVAL_LAB",
        "schema": "PUBLIC",
    }
    session = Session.builder.configs(connection_params).create()if os.environ.get("SNOWFLAKE_ACCOUNT") is None or os.environ.get("SNOWFLAKE_USER") is None or os.environ.get("SNOWFLAKE_PASSWORD") is None:
    raise ValueError(
        "Missing required environment variables. Please set SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER, and SNOWFLAKE_PASSWORD."
    )

# Set context
session.sql("USE DATABASE CMO_EVAL_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE CMO_EVAL_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Database: {session.sql('SELECT CURRENT_DATABASE()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

AttributeError: 'NoneType' object has no attribute 'find'

---
## Section 1: Build the Agent

We'll create a Cortex Agent with three tools:
- **Cortex Analyst** — queries structured campaign performance data via a semantic view
- **Cortex Search** — retrieves marketing strategy documents via RAG
- **Agent Skill** — formats responses as executive summaries

In [ ]:
# Create the CMO Assistant agent
session.sql("""
CREATE OR REPLACE AGENT CMO_ASSISTANT
  COMMENT = 'Marketing/Finance assistant for campaign performance and strategy'
FROM SPECIFICATION
$$
models:
  orchestration: auto

instructions:
  response: |
    You are a CMO assistant that helps marketing leaders understand campaign performance,
    budget allocation, and strategic recommendations. Be concise and data-driven.
    When presenting financial data, always include the time period and round to 2 decimal places.
  orchestration: |
    For quantitative questions about spend, revenue, ROI, conversions, or performance metrics, use the campaign_analytics tool.
    For questions about strategy, methodology, planning documents, or guidelines, use the strategy_search tool.
    For requests to summarize or create executive briefs, first gather data with the appropriate tool, then use executive_summary to format.

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "campaign_analytics"
      description: "Query structured campaign performance data including spend, revenue, impressions, clicks, conversions, ROI, CPC, and CPA by channel and time period. Use for any quantitative marketing or financial question about campaign performance."
  - tool_spec:
      type: "cortex_search"
      name: "strategy_search"
      description: "Search marketing strategy documents, budget allocation methodology, attribution models, performance benchmarks, and planning briefs. Use for qualitative questions about strategy, process, methodology, or guidelines."
  - tool_spec:
      type: "generic"
      name: "executive_summary"
      description: "Format data and insights into a concise executive summary suitable for C-suite presentation. Use after gathering data when the user asks for a summary, brief, or executive-level view."

tool_resources:
  campaign_analytics:
    semantic_view: "CMO_EVAL_LAB.PUBLIC.CMO_ANALYTICS"
  strategy_search:
    name: "CMO_EVAL_LAB.PUBLIC.STRATEGY_SEARCH_SVC"
    max_results: "3"
  executive_summary:
    instructions: |
      Format the response as a brief executive summary with:
      - A one-line headline insight
      - 3-5 bullet points with key findings
      - A recommended action
      Keep total length under 200 words.
$$
""").collect()

print("Agent CMO_ASSISTANT created successfully.")

In [ ]:
# Quick smoke test — verify the agent responds
result = session.sql("""
SELECT SNOWFLAKE.CORTEX.AGENT(
  'CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT',
  'What was our total spend in 2024?'
) AS response
""").collect()

print("Agent response:")
print(result[0]['RESPONSE'][:500])

---
## Section 2: Build the Evaluation Dataset

A good evaluation dataset covers the full intent distribution:
- **Happy paths** — common questions the agent should handle well
- **Multi-tool queries** — questions requiring coordination between tools
- **Edge cases** — specific, narrow queries
- **Out-of-scope** — questions the agent should refuse

We'll create 12 questions with ground truth that defines what a correct response looks like.

In [ ]:
# Create the evaluation questions table
session.sql("""
CREATE OR REPLACE TABLE EVAL_QUESTIONS (
    INPUT_QUERY   VARCHAR,
    GROUND_TRUTH  VARIANT
)
""").collect()

# Insert evaluation questions with ground truth
questions = [
    # 1. Happy path — Analyst
    (
        "What was total spend across all channels in 2024?",
        '{"ground_truth_output": "Total marketing spend across all channels in 2024 was approximately $1,726,500. The response should present this as a single aggregate number covering all four channels (Paid Search, Social Media, Email, Display) for the full calendar year 2024. Values within ±1% are acceptable."}'
    ),
    # 2. Happy path — Analyst
    (
        "Which channel had the highest ROI in Q4 2024?",
        '{"ground_truth_output": "Email had the highest ROI in Q4 2024. Email ROI was approximately 11.3x (or ROAS of 12.3x), significantly outperforming other channels. The response should identify Email as the top performer and provide the ROI/ROAS figure. It should scope the answer explicitly to Q4 (October-December 2024)."}'
    ),
    # 3. Happy path — Search
    (
        "What is our attribution methodology?",
        '{"ground_truth_output": "The response should describe a data-driven multi-touch attribution model with a 30-day lookback window. Key details: first-touch weight 0.2, last-touch weight 0.3, middle touches share remaining 0.5 proportional to recency. It should mention that data refreshes daily with a 48-hour lag. The response should NOT fabricate details not in the strategy docs."}'
    ),
    # 4. Multi-tool — Analyst + Skill
    (
        "Give me an executive summary of Q4 campaign performance for the board.",
        '{"ground_truth_output": "The response should combine quantitative Q4 data (total spend ~$590K, total revenue ~$2.06M, overall ROAS ~3.5x) with executive formatting: a headline insight, bullet points with key findings by channel, and a recommended action. It should be concise (under 200 words) and suitable for board-level communication."}'
    ),
    # 5. Multi-tool — Analyst + Search
    (
        "How does our actual Q4 spend allocation compare to what our channel strategy recommends?",
        '{"ground_truth_output": "The response should compare actual Q4 spend percentages by channel against the strategy document recommendations (Paid Search 40%, Social Media 25%, Email 10%, Display 15%, Flex 10%). It should calculate actual percentages from the spend data and note any significant deviations. This requires both querying spend data AND retrieving the channel strategy document."}'
    ),
    # 6. Edge case — Analyst (specific)
    (
        "What was our CPA for email in March 2024?",
        '{"ground_truth_output": "The CPA for Email in March 2024 was approximately $8.04 (calculated as $9,000 spend / 1,120 conversions). The response must be scoped to exactly March 2024 and the Email channel only. Values within ±$0.50 are acceptable."}'
    ),
    # 7. Happy path — Search
    (
        "What are our brand guidelines for reporting financial metrics?",
        '{"ground_truth_output": "The response should reference the brand guidelines document and include rules such as: currency in USD rounded to 2 decimal places, percentages to 1 decimal place, ROI expressed as a multiplier (e.g., 3.2x not 320%), time periods must be explicitly stated, and executive summaries must lead with the most impactful insight. It should NOT invent guidelines not in the documents."}'
    ),
    # 8. Complex — Analyst
    (
        "Compare paid search vs social media ROI trend over H2 2024 (July through December).",
        '{"ground_truth_output": "The response should show monthly or quarterly ROI for both Paid Search and Social Media during July-December 2024. Paid Search ROI should be significantly higher than Social Media throughout H2. Paid Search ROAS ranges roughly 3.3x-4.1x while Social Media ranges roughly 1.9x-2.1x. The trend should show both channels improving in Q4 due to holiday seasonality."}'
    ),
    # 9. Out-of-scope — Refusal
    (
        "What's the weather forecast for tomorrow?",
        '{"ground_truth_output": "The response should clearly state that weather information is outside the agents capabilities and ideally redirect to what it can help with (marketing performance, campaign data, strategy questions). It should NOT fabricate a weather forecast or attempt to answer."}'
    ),
    # 10. Multi-tool — Analyst + Skill
    (
        "Create an executive brief on our highest-performing channel for 2024.",
        '{"ground_truth_output": "The response should identify Email as the highest-performing channel by ROI (approximately 10.5x ROAS for the year) and present the finding in executive summary format: headline insight, bullet points with supporting metrics (total Email spend ~$131.5K, revenue ~$1.46M, conversion rate ~7%), and a recommended action. Alternatively, Paid Search could be identified as highest by absolute revenue."}'
    ),
    # 11. Happy path — Analyst
    (
        "What campaigns did we run in Q1 2024?",
        '{"ground_truth_output": "The response should list the Q1 campaign name: Q1 Brand Awareness, which ran across all four channels (Paid Search, Social Media, Email, Display) during January-March 2024. It should scope the answer to Q1 only and not include campaigns from other quarters."}'
    ),
    # 12. Out-of-scope — Refusal
    (
        "What did our competitors spend on marketing last year?",
        '{"ground_truth_output": "The response should state that competitive spend data is not available in the system. It should NOT fabricate competitor data. It may offer to help with the companys own marketing spend data as an alternative."}'
    ),
]

# Insert all questions
for query, ground_truth in questions:
    escaped_query = query.replace("'", "''")
    session.sql(f"""
        INSERT INTO EVAL_QUESTIONS
        SELECT '{escaped_query}', PARSE_JSON('{ground_truth}')
    """).collect()

print(f"Inserted {len(questions)} evaluation questions.")
session.sql("SELECT INPUT_QUERY, GROUND_TRUTH:ground_truth_output::VARCHAR AS EXPECTED FROM EVAL_QUESTIONS LIMIT 3").show()

In [ ]:
# Register as an evaluation dataset
session.sql("""
CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'CMO_EVAL_LAB.PUBLIC.EVAL_QUESTIONS',
  'CMO_EVAL_LAB.PUBLIC.CMO_EVAL_DATASET',
  OBJECT_CONSTRUCT(
    'query_text', 'INPUT_QUERY',
    'expected_tools', 'GROUND_TRUTH'
  )
)
""").collect()

print("Dataset registered: CMO_EVAL_DATASET")
session.sql("SHOW DATASETS IN SCHEMA CMO_EVAL_LAB.PUBLIC").show()

---
## Section 3: Define Metrics and Run the Evaluation

We'll use three metrics:
- **answer_correctness** (built-in) — compares agent output to ground truth
- **logical_consistency** (built-in, reference-free) — checks internal consistency of planning/execution
- **tool_selection** (custom) — evaluates whether the agent chose the right tool for each query

In [ ]:
# Write the evaluation config YAML
eval_config_yaml = """
# Cortex Agent Evaluation Configuration
# CMO Assistant — Baseline Run

evaluation:
  agent_params:
    agent_name: "CMO_ASSISTANT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "CMO Assistant evaluation"
  source_metadata:
    type: "dataset"
    dataset_name: "CMO_EVAL_DATASET"

metrics:
  # Built-in: compare agent output to ground truth
  - "answer_correctness"
  # Built-in: reference-free consistency check
  - "logical_consistency"
  # Custom: evaluate tool selection quality
  - name: "tool_selection"
    score_ranges:
      min_score: [1, 3]
      median_score: [4, 6]
      max_score: [7, 10]
    prompt: |
      Evaluate whether the agent selected the correct tool(s) for the user's query.

      User query: {{input}}
      Tools used: {{tool_info}}
      Expected behavior: {{ground_truth}}
      Agent response: {{output}}

      Rate from 1-10:
      1-3 = Wrong tool selected, unnecessary tool calls, or failed to use a tool when one was needed
      4-6 = Partially correct tool selection (used the right primary tool but missed a secondary tool, or made redundant calls)
      7-10 = Optimal tool selection for the query intent

      Evaluation criteria:
      - Did the agent use campaign_analytics for quantitative questions about spend, revenue, ROI, etc.?
      - Did it use strategy_search for qualitative questions about methodology, strategy, or guidelines?
      - Did it correctly combine tools for multi-faceted questions?
      - For out-of-scope questions, did it avoid calling tools and instead refuse gracefully?

      Provide your score as a single integer.
""".strip()

# Write YAML to a local temp file, then PUT to stage
import tempfile

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(eval_config_yaml)
    yaml_path = f.name

session.sql(f"""
PUT 'file://{yaml_path}' @CMO_EVAL_LAB.PUBLIC.EVAL_STAGE
  AUTO_COMPRESS=FALSE
  OVERWRITE=TRUE
""").collect()

print("Evaluation config uploaded to @EVAL_STAGE")
session.sql("LIST @CMO_EVAL_LAB.PUBLIC.EVAL_STAGE").show()

In [ ]:
# Start the baseline evaluation run
session.sql("""
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'baseline-v1'),
  '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
)
""").collect()

print("Evaluation 'baseline-v1' started. This will take 2-5 minutes...")

In [ ]:
# Poll evaluation status — re-run this cell until STATUS shows COMPLETED
import time

for i in range(20):
    status = session.sql("""
    CALL EXECUTE_AI_EVALUATION(
      'STATUS',
      OBJECT_CONSTRUCT('run_name', 'baseline-v1'),
      '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
    )
    """).collect()
    
    current_status = status[0]['STATUS'] if 'STATUS' in status[0].as_dict() else str(status[0])
    print(f"[{i+1}/20] Status: {current_status}")
    
    if 'COMPLETED' in str(current_status).upper():
        print("\nEvaluation complete!")
        break
    
    time.sleep(30)
else:
    print("\nStill running — re-run this cell to check again.")

---
## Section 4: Inspect Results and Iterate

Now we'll look at the evaluation results, identify the weakest areas, and make a targeted improvement to the agent's instructions.

In [ ]:
# View overall scores by metric
session.sql("""
SELECT
    METRIC_NAME,
    METRIC_TYPE,
    COUNT(*) AS NUM_RECORDS,
    ROUND(AVG(EVAL_AGG_SCORE), 3) AS AVG_SCORE,
    ROUND(MIN(EVAL_AGG_SCORE), 3) AS MIN_SCORE,
    ROUND(MAX(EVAL_AGG_SCORE), 3) AS MAX_SCORE
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
))
GROUP BY METRIC_NAME, METRIC_TYPE
ORDER BY AVG_SCORE ASC
""").show()

In [ ]:
# View per-question detail — sorted by lowest scores to find failures
session.sql("""
SELECT
    INPUT,
    METRIC_NAME,
    ROUND(EVAL_AGG_SCORE, 3) AS SCORE,
    METRIC_CALLS[0]:explanation::VARCHAR AS EXPLANATION
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
))
ORDER BY EVAL_AGG_SCORE ASC
LIMIT 10
""").show(max_width=120)

In [ ]:
# Look at multi-tool and out-of-scope questions specifically
session.sql("""
SELECT
    INPUT,
    METRIC_NAME,
    ROUND(EVAL_AGG_SCORE, 3) AS SCORE,
    LEFT(OUTPUT, 200) AS AGENT_RESPONSE_PREVIEW
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
))
WHERE INPUT ILIKE '%executive%' 
   OR INPUT ILIKE '%competitor%' 
   OR INPUT ILIKE '%weather%'
   OR INPUT ILIKE '%compare%allocation%'
ORDER BY INPUT, METRIC_NAME
""").show(max_width=120)

### Iterate: Improve the Agent

Based on the results above, we'll make a targeted fix. Common failure modes for this agent include:
- **Multi-tool coordination** — the agent may not combine analytics + search or analytics + summary tools effectively
- **Out-of-scope handling** — the agent may attempt to answer questions it should refuse

We'll update the orchestration instructions to be more explicit about these scenarios.

In [ ]:
# Improve the agent's instructions based on evaluation findings
session.sql("""
ALTER AGENT CMO_ASSISTANT SET
  INSTRUCTIONS = '
response: |
  You are a CMO assistant that helps marketing leaders understand campaign performance,
  budget allocation, and strategic recommendations. Be concise and data-driven.
  When presenting financial data, always include the time period and round to 2 decimal places.
  Express ROI as a multiplier (e.g., 3.2x).
orchestration: |
  TOOL SELECTION RULES (follow strictly):
  1. QUANTITATIVE questions (spend, revenue, ROI, CPA, conversions, performance metrics) -> use campaign_analytics
  2. QUALITATIVE questions (strategy, methodology, guidelines, planning, benchmarks) -> use strategy_search
  3. COMPARISON questions that reference both data AND strategy documents -> use campaign_analytics FIRST, then strategy_search
  4. EXECUTIVE SUMMARY requests -> gather data with the appropriate tool first, then use executive_summary to format
  5. OUT-OF-SCOPE questions (weather, competitors, anything not about our marketing data or strategy) -> DO NOT call any tool. Politely decline and explain what you CAN help with.

  MULTI-TOOL COORDINATION:
  - When a question requires both quantitative data and qualitative context, ALWAYS call both tools.
  - Example: "How does our spend compare to strategy recommendations?" requires campaign_analytics for actual spend AND strategy_search for the recommended allocation.
  - For executive briefs, ALWAYS gather data before formatting with executive_summary.
'
""").collect()

print("Agent instructions updated with improved orchestration guidance.")

In [ ]:
# Commit the improved version as an immutable snapshot
session.sql("""
ALTER AGENT CMO_ASSISTANT COMMIT
  COMMENT = 'Improved multi-tool orchestration and explicit out-of-scope handling'
""").collect()

print("Version committed. Running improved evaluation...")

# Start the second evaluation run
session.sql("""
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'improved-v2'),
  '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
)
""").collect()

print("Evaluation 'improved-v2' started. This will take 2-5 minutes...")

In [ ]:
# Poll status for improved run
import time

for i in range(20):
    status = session.sql("""
    CALL EXECUTE_AI_EVALUATION(
      'STATUS',
      OBJECT_CONSTRUCT('run_name', 'improved-v2'),
      '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
    )
    """).collect()
    
    current_status = status[0]['STATUS'] if 'STATUS' in status[0].as_dict() else str(status[0])
    print(f"[{i+1}/20] Status: {current_status}")
    
    if 'COMPLETED' in str(current_status).upper():
        print("\nEvaluation complete!")
        break
    
    time.sleep(30)
else:
    print("\nStill running — re-run this cell to check again.")

In [ ]:
# Compare baseline vs improved — side by side
session.sql("""
WITH baseline AS (
    SELECT METRIC_NAME, ROUND(AVG(EVAL_AGG_SCORE), 3) AS BASELINE_AVG
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
    ))
    GROUP BY METRIC_NAME
),
improved AS (
    SELECT METRIC_NAME, ROUND(AVG(EVAL_AGG_SCORE), 3) AS IMPROVED_AVG
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'improved-v2'
    ))
    GROUP BY METRIC_NAME
)
SELECT
    b.METRIC_NAME,
    b.BASELINE_AVG,
    i.IMPROVED_AVG,
    ROUND(i.IMPROVED_AVG - b.BASELINE_AVG, 3) AS DELTA
FROM baseline b
JOIN improved i ON b.METRIC_NAME = i.METRIC_NAME
ORDER BY DELTA DESC
""").show()

---
## Section 5: Version and Promote

With the improved version showing better scores, we'll promote it to production using the alias system. This means application code that references the `production` alias will automatically pick up the improved version — no code changes required.

In [ ]:
# View all versions of the agent
session.sql("SHOW VERSIONS IN AGENT CMO_ASSISTANT").show()

In [ ]:
# Promote the improved version to production
# The LAST committed version is our improved one
session.sql("""
ALTER AGENT CMO_ASSISTANT MODIFY VERSION LAST SET ALIAS = production
""").collect()

print("Promoted improved version to 'production' alias.")

# Verify the alias assignment
session.sql("SHOW VERSIONS IN AGENT CMO_ASSISTANT").show()

In [ ]:
# Test the production alias — this is how your application would call the agent
result = session.sql("""
SELECT SNOWFLAKE.CORTEX.AGENT(
  'CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT:production',
  'Which channel should we increase budget for next quarter based on 2024 performance?'
) AS response
""").collect()

print("Production alias response:")
print(result[0]['RESPONSE'][:600])

---
## Summary: What You Accomplished

In this lab you completed the full Cortex Agent evaluation lifecycle:

| Step | What you did |
|------|-------------|
| **Build** | Created a multi-tool agent (Analyst + Search + Skill) |
| **Dataset** | Designed 12 eval questions covering happy paths, multi-tool, edge cases, and refusals |
| **Metrics** | Used 2 built-in metrics + 1 custom `tool_selection` metric |
| **Evaluate** | Ran a baseline evaluation and inspected per-question scores and explanations |
| **Iterate** | Identified weak areas and improved orchestration instructions |
| **Compare** | Ran a second evaluation and compared scores side-by-side |
| **Promote** | Committed the improved version and assigned the `production` alias |

### What's Next?

- **Expand the dataset** — Add more questions from real user interactions as they become available
- **Add CI/CD gates** — Wrap `EXECUTE_AI_EVALUATION` in a Task or CI pipeline with threshold checks
- **Monitor production** — Query `SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS` for ongoing correctness signals
- **Iterate further** — Test different orchestration models, refine tool descriptions, tune the semantic view

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS CMO_EVAL_LAB CASCADE").collect()
# print("Lab resources cleaned up.")